# Ham ya da Spam?

🎯 Bu görevin amacı, e-postaları **spam (1)** veya **normal e-posta (0)** olarak sınıflandırmaktır.

🧹 İlk olarak, bu metin verilerine **temizleme (cleaning)** teknikleri uygulanacaktır.

👩🏻‍🔬 Ardından, temizlenmiş metinler **sayısal bir gösterime** dönüştürülecektir.

✉️ Son olarak, her bir e-postayı spam mı yoksa normal mi olduğunu sınıflandırmak için  
***Multinomial Naive Bayes*** modeli uygulanacaktır.


## (0) NTLK kütüphanesi (Doğal Dil Araç Seti)

In [3]:
!pip install nltk


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python -m pip install --upgrade pip


In [5]:
# nltk'yi ilk kez içe aktarırken, birkaç yerleşik kütüphaneyi de indirmemiz gerekir.

import nltk

nltk.download('stopwords')
nltk.download('punkt')      # nltk<3.9.0 için
nltk.download('punkt_tab')  # nltk>=3.9.0 için
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package stopwords to /home/semih/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /home/semih/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /home/semih/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package wordnet to /home/semih/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /home/semih/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [6]:
import pandas as pd

df = pd.read_csv("https://d32aokrjazspmn.cloudfront.net/materials/ham_spam_emails.csv")
df.head()

,text,spam
0,Subject: naturally irresistible your corporate...,1
1,Subject: the stock trading gunslinger fanny i...,1
2,Subject: unbelievable new homes made easy im ...,1
3,Subject: 4 color printing special request add...,1
4,"Subject: do not have money , get software cds ...",1


## (1) (Metin) veri setinin temizlenmesi

Veri kümesi, ham [0] veya spam [1] olarak sınıflandırılan e-postalardan oluşur. Tahmin modelini eğitmeden önce veri kümesini temizlemeniz gerekir.

### (1.1) Noktalama İşaretlerini Kaldır

❓ Noktalama işaretlerini kaldırmak için bir işlev oluşturun. Bunu `text` sütununa uygulayın ve çıktıyı `clean_text` adlı veri çerçevesinin yeni bir sütununa ekleyin. ❓

In [7]:
import string

def remove_punctuation(text):
    # Her bir karakteri kontrol et, noktalama işareti değilse birleştir
    return "".join([char for char in text if char not in string.punctuation])

# Fonksiyonu 'text' sütununa uygula ve yeni sütunu oluştur
df['clean_text'] = df['text'].apply(remove_punctuation)

# Sonucu kontrol et
df.head()

,text,spam,clean_text
0,Subject: naturally irresistible your corporate...,1,Subject naturally irresistible your corporate ...
1,Subject: the stock trading gunslinger fanny i...,1,Subject the stock trading gunslinger fanny is...
2,Subject: unbelievable new homes made easy im ...,1,Subject unbelievable new homes made easy im w...
3,Subject: 4 color printing special request add...,1,Subject 4 color printing special request addi...
4,"Subject: do not have money , get software cds ...",1,Subject do not have money get software cds fr...


### (1.2) Küçük Harf

❓ Metni küçük harfe çeviren bir işlev oluşturun. Bunu `clean_text`'e uygulayın ❓

In [8]:
# clean_text sütunundaki tüm metinleri küçük harfe çeviren fonksiyon
def to_lowercase(text):
    return text.lower()

# Fonksiyonu mevcut clean_text sütununa uygula
df['clean_text'] = df['clean_text'].apply(to_lowercase)

# Değişikliği kontrol et
df.head()

,text,spam,clean_text
0,Subject: naturally irresistible your corporate...,1,subject naturally irresistible your corporate ...
1,Subject: the stock trading gunslinger fanny i...,1,subject the stock trading gunslinger fanny is...
2,Subject: unbelievable new homes made easy im ...,1,subject unbelievable new homes made easy im w...
3,Subject: 4 color printing special request add...,1,subject 4 color printing special request addi...
4,"Subject: do not have money , get software cds ...",1,subject do not have money get software cds fr...


### (1.3) Sayıları Kaldır

❓ Metinden sayıları kaldırmak için bir işlev oluşturun. Bunu `clean_text`'e uygulayın ❓

In [9]:
def remove_numbers(text):
    # Karakter sayı değilse (isdigit() False ise) sakla ve birleştir
    return "".join([char for char in text if not char.isdigit()])

# Fonksiyonu clean_text sütununa uygula
df['clean_text'] = df['clean_text'].apply(remove_numbers)

# Sonucu kontrol et
df.head()

,text,spam,clean_text
0,Subject: naturally irresistible your corporate...,1,subject naturally irresistible your corporate ...
1,Subject: the stock trading gunslinger fanny i...,1,subject the stock trading gunslinger fanny is...
2,Subject: unbelievable new homes made easy im ...,1,subject unbelievable new homes made easy im w...
3,Subject: 4 color printing special request add...,1,subject color printing special request addit...
4,"Subject: do not have money , get software cds ...",1,subject do not have money get software cds fr...


### (1.4) StopWords'ü kaldırın

❓ Metinden durdurma kelimelerini kaldırmak için bir işlev oluşturun. Bunu `clean_text`'e uygulayın. ❓

In [10]:
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

# İngilizce durdurma kelimeleri listesini alalım
stop_words = set(stopwords.words('english'))

def remove_stopwords(text):
    # 1. Metni kelimelerine ayır (tokenize)
    word_tokens = word_tokenize(text)
    
    # 2. Stopwords listesinde olmayan kelimeleri filtrele
    filtered_text = [word for word in word_tokens if word not in stop_words]
    
    # 3. Kelimeleri tekrar birleştirerek metne dönüştür
    return " ".join(filtered_text)

# Fonksiyonu clean_text sütununa uygula
df['clean_text'] = df['clean_text'].apply(remove_stopwords)

# Sonucu kontrol et
df.head()

,text,spam,clean_text
0,Subject: naturally irresistible your corporate...,1,subject naturally irresistible corporate ident...
1,Subject: the stock trading gunslinger fanny i...,1,subject stock trading gunslinger fanny merrill...
2,Subject: unbelievable new homes made easy im ...,1,subject unbelievable new homes made easy im wa...
3,Subject: 4 color printing special request add...,1,subject color printing special request additio...
4,"Subject: do not have money , get software cds ...",1,subject money get software cds software compat...


### (1.5) Lemmatize

❓ Metni lemmatize etmek için bir fonksiyon oluşturun. Çıktının bir kelime listesi değil, tek bir dize olduğundan emin olun. Bunu `clean_text`'e uygulayın. ❓

In [11]:
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

# Lemmatizer nesnesini oluşturalım
lemmatizer = WordNetLemmatizer()

def lemmatize_text(text):
    # 1. Metni kelimelere ayır
    words = word_tokenize(text)
    
    # 2. Her kelimenin köküne (lemma) dön ve listeye al
    lemmatized_words = [lemmatizer.lemmatize(word) for word in words]
    
    # 3. Kelime listesini tekrar tek bir dize (string) olarak birleştir
    return " ".join(lemmatized_words)

# Fonksiyonu uygula
df['clean_text'] = df['clean_text'].apply(lemmatize_text)

# Sonucu kontrol et
df.head()

,text,spam,clean_text
0,Subject: naturally irresistible your corporate...,1,subject naturally irresistible corporate ident...
1,Subject: the stock trading gunslinger fanny i...,1,subject stock trading gunslinger fanny merrill...
2,Subject: unbelievable new homes made easy im ...,1,subject unbelievable new home made easy im wan...
3,Subject: 4 color printing special request add...,1,subject color printing special request additio...
4,"Subject: do not have money , get software cds ...",1,subject money get software cd software compati...


## (2) Bag-of-Words Modellemesi

### (2.1) Metin verilerini sayılara dönüştürme

❓ `clean_text`'i varsayılan CountVectorizer ile Bag-of-Words temsiline vektörleştirin. `X_bow` olarak kaydedin. ❓

In [12]:
from sklearn.feature_extraction.text import CountVectorizer

# 1. CountVectorizer nesnesini başlatalım
vectorizer = CountVectorizer()

# 2. Temizlenmiş metinleri (clean_text) vektörleştirelim
# fit_transform hem kelime dağarcığını öğrenir hem de veriyi dönüştürür
X_bow = vectorizer.fit_transform(df['clean_text'])

# Sonucu kontrol edelim (kaç e-posta ve toplam kaç benzersiz kelime var?)
print(f"X_bow boyutu: {X_bow.shape}")

X_bow boyutu: (5728, 30988)


### (2.2) Çok terimli Naive Bayes Modellemesi

❓ MultinomialNB modelini bag-of-words verileriyle çapraz doğrulayın. Modelin doğruluğunu puanlayın. ❓

In [14]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import cross_val_score

# 1. Multinomial Naive Bayes modelini tanımlayalım
nb_model = MultinomialNB()

# 2. Hedef değişkenimizi (y) tanımlayalım
y = df['spam']

# 3. Modeli X_bow verisiyle çapraz doğrulayalım (5 katlı/cv=5)
# Bu işlem veriyi 5 parçaya böler ve her seferinde farklı bir parçayı test için ayırır
cv_scores = cross_val_score(nb_model, X_bow, y, cv=5)

# 4. Sonuçları yazdıralım
print(f"Her bir katman için doğruluk skorları: {cv_scores}")
print(f"Ortalama Doğruluk Skoru: {cv_scores.mean():.4f}")

Her bir katman için doğruluk skorları: [0.98691099 0.9895288  0.991274   0.98777293 0.99213974]
Ortalama Doğruluk Skoru: 0.9895


🏁 Tebrikler!

💾 Not defterinizi git add/commit/push yapmayı unutmayın...

🚀 ... ve bir sonraki challenge'a geçin!